# Phase 3: Hybrid Fusion Model — Training & Evaluation

This notebook trains and evaluates the **Hybrid Attention-Weighted Fusion Model (Model C)**.

### ⚡ Crash-Safe Training
Training saves a **full checkpoint every epoch** (model + optimizer + scheduler state).
If the notebook disconnects:
1. Re-run the setup cells (0-1)
2. Run Section 4 — it will **automatically resume** from the last completed epoch

### 🚀 High-Speed Training
Run **Section 2.5 (Pre-compute Features)** before starting training. This extracts texture features in parallel and saves them to your Drive, making training 50x faster.

## 0. Environment Setup

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    IN_COLAB = True
    print('✅ Google Drive mounted')
except ImportError:
    IN_COLAB = False
    print('ℹ️  Not in Colab')

In [ ]:
import sys, os
from pathlib import Path

colab_path = Path('/content/drive/MyDrive/Hybrid-Dermatologist')
project_root = colab_path if colab_path.exists() else Path(os.getcwd()).resolve()
if project_root.name == 'phase3': project_root = project_root.parents[1]

os.chdir(str(project_root))
if str(project_root) not in sys.path: sys.path.insert(0, str(project_root))
print(f'✅ Working directory: {os.getcwd()}')

In [ ]:
import torch, numpy as np, matplotlib.pyplot as plt, pandas as pd
from src.skin_analysis.phase3.config import Phase3Config
from src.skin_analysis.phase3.model_c import HybridFusionModel
from src.skin_analysis.phase3.dataset import build_hybrid_dataloaders, compute_class_weights
from src.skin_analysis.phase3.train_c import train_hybrid, seed_everything, detect_device

seed_everything(42)
device = detect_device()
cfg = Phase3Config()
print(f'Device: {device}')

## 1. Check Training Status

In [ ]:
ckpt_path = cfg.output_dir / 'checkpoint_hybrid.pth'
if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'  ↻ EXISTING CHECKPOINT FOUND!')
    print(f'    Stage: {ckpt.get("stage")} | Epoch: {ckpt.get("epoch")} | Best F1: {ckpt.get("best_f1", 0):.4f}')
else:
    print('  No checkpoint found — starting from scratch.')

## 2. Data Loading

In [ ]:
train_loader, val_loader, train_dataset, val_dataset = build_hybrid_dataloaders(cfg)
print(f'Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}')

## 2.5. Pre-compute ML Features (RUN THIS FOR FAST TRAINING)

This script extracts LBP/GLCM texture features in parallel. **Highly recommended on Colab.**

In [ ]:
!python3 scripts/precompute_features.py

## 3. Model & Training

In [ ]:
class_weights = compute_class_weights(train_dataset)
model = HybridFusionModel.from_phase2_checkpoint(
    checkpoint_path=cfg.phase2_checkpoint,
    num_classes=cfg.num_classes,
    ml_feature_dim=cfg.ml_feature_dim,
    fusion_hidden_dim=cfg.fusion_hidden_dim,
    device=device,
)

history = train_hybrid(model, train_loader, val_loader, cfg, class_weights=class_weights, device=device)

## 4. Results & Learning Curves

In [ ]:
if history:
    df = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    df.plot(x='epoch', y=['train_loss', 'val_loss'], ax=axes[0], title='Loss')
    df.plot(x='epoch', y=['val_f1_weighted'], ax=axes[1], title='F1 Score')
    plt.show()